# 📋 GUÍA DE USO - SCRAPING COPPEL LIMPIO

## 🚀 INSTRUCCIONES DE EJECUCIÓN

### ✅ EJECUTAR ESTAS 4 CELDAS EN ORDEN:

1. **📦 CELDA 1:** Importaciones y funciones básicas
2. **⚡ CELDA 2:** Función de scraping con filtro de fecha del día
3. **🎯 CELDA 3:** Ejecutar scraping solo de productos del día actual
4. **💾 CELDA 4:** Generar CSV con productos únicamente de hoy

### 📅 CONFIGURACIÓN:
- **Filtro de fecha:** Solo productos del día actual (2025-09-12)
- **Procesamiento:** Paralelo para máxima velocidad
- **Salida:** CSV solo con productos extraídos hoy

### 📊 RESULTADO:
- Archivo CSV con productos del día de hoy únicamente
- Datos limpios y organizados
- Listo para análisis

---

# 🛒 SCRAPER DE COPPEL - CELULARES
## Versión Optimizada y Limpia

Este notebook contiene el código optimizado para hacer scraping de todos los productos de celulares en Coppel.com

### 📋 Instrucciones de uso:
1. **Ejecutar Celda 1:** Configuración e importaciones
2. **Ejecutar Celda 2:** Funciones de scraping 
3. **Ejecutar Celda 3:** Scraping paralelo completo
4. **Ejecutar Celda 4:** Generar CSV con resultados

### 📊 Resultado esperado:
- **~4,400+ productos** extraídos de todas las categorías
- **Procesamiento paralelo** para máxima velocidad
- **CSV completo** con toda la información
- **Todas las páginas** de todas las categorías

In [1]:
# 📦 CELDA 1: CONFIGURACIÓN E IMPORTACIONES
print("🚀 Inicializando Scraper de Coppel...")

# Importaciones necesarias
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import pandas as pd
import concurrent.futures
from threading import Lock
from datetime import datetime
import re
import unicodedata

# Configuración de Chrome
options = Options()
#options.add_argument("--headless") #
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

def normalizar_sku(valor):
    """Convierte el SKU a texto sin decimales (747246874.0 -> '747246874')"""
    if pd.isna(valor):
        return ""
    s = str(valor).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    return s

# Cargar diccionarios de colores, capacidades y marcas
def cargar_diccionarios():
    """Carga los diccionarios de colores, capacidades y marcas desde Excel"""
    try:
        # Cargar archivo Excel
        ruta_diccionario = r"C:\Users\c90029859\Documents\diccionario2.xlsx"
        df_datos = pd.read_excel(ruta_diccionario, sheet_name="Coppel")
        
        # Limpiar datos
        df_datos = df_datos.dropna(subset=['SKU'])
        df_datos['SKU'] = df_datos['SKU'].apply(normalizar_sku)
        
        # Colores
        sku_color_dict = {}
        if 'Color' in df_datos.columns:
            df_colores = df_datos.dropna(subset=['Color'])
            sku_color_dict = dict(zip(df_colores['SKU'], df_colores['Color'].astype(str)))
        
        # Capacidades  
        sku_capacidad_dict = {}
        if 'Capacidad' in df_datos.columns:
            df_capacidades = df_datos.dropna(subset=['Capacidad'])
            sku_capacidad_dict = dict(zip(df_capacidades['SKU'], df_capacidades['Capacidad'].astype(str)))
        
        # Marcas
        sku_marca_dict = {}
        if 'Marca' in df_datos.columns:
            df_marcas = df_datos.dropna(subset=['Marca'])
            sku_marca_dict = dict(zip(df_marcas['SKU'], df_marcas['Marca'].astype(str)))
        
        print(f"✅ Diccionarios cargados:")
        print(f"   • Colores: {len(sku_color_dict)} registros")
        print(f"   • Capacidades: {len(sku_capacidad_dict)} registros") 
        print(f"   • Marcas: {len(sku_marca_dict)} registros")
        
        return sku_color_dict, sku_capacidad_dict, sku_marca_dict
    except Exception as e:
        print(f"⚠️ Error cargando diccionarios: {e}")
        return {}, {}, {}

# Cargar diccionarios
sku_color_dict, sku_capacidad_dict, sku_marca_dict = cargar_diccionarios()

print("✅ Configuración completada")
print("▶️ Continúa con la Celda 2")

🚀 Inicializando Scraper de Coppel...
✅ Diccionarios cargados:
   • Colores: 14398 registros
   • Capacidades: 17277 registros
   • Marcas: 17402 registros
✅ Configuración completada
▶️ Continúa con la Celda 2


In [2]:
# 🔧 CELDA 2: FUNCIONES PRINCIPALES DE SCRAPING
print("🔧 Cargando funciones de scraping...")

# Funciones auxiliares
def format_price(price_raw):
    """Limpia y formatea precios"""
    if not price_raw:
        return ""
    return price_raw.replace('$', '').replace(',', '').strip()

def extraer_sku(url):
    """Extrae SKU de la URL"""
    match = re.search(r"(?:mkp|pm)-(\d+)", url)
    return match.group(1) if match else ""

def calcular_porcentaje_descuento(precio_normal, precio_oferta):
    """Calcula el porcentaje de descuento entre precio normal y precio de oferta"""
    try:
        if not precio_oferta or precio_oferta == "":
            return "No discount"
        
        p_normal = float(precio_normal.replace('$','').replace(',','')) if precio_normal else 0
        p_oferta = float(precio_oferta.replace('$','').replace(',','')) if precio_oferta else 0
        
        if p_normal > 0 and p_oferta > 0 and p_normal > p_oferta:
            descuento = round(100 * (p_normal - p_oferta) / p_normal, 2)
            return f"{descuento}%"
        elif p_oferta > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_marca(sku, nombre=""):
    """Extrae marca usando diccionario Excel primero, luego del nombre"""
    # Prioridad 1: Buscar en diccionario por SKU
    sku = normalizar_sku(sku)
    if sku and sku in sku_marca_dict:
        marca_excel = sku_marca_dict[sku]
        if marca_excel and marca_excel.strip() and marca_excel.strip().lower() not in ['nan', 'none', '']:
            return marca_excel.strip().upper()
    
    # Prioridad 2: Extraer del nombre del producto
    if not nombre:
        return ""
    
    nombre_upper = nombre.upper()
    marcas = ["APPLE", "SAMSUNG", "MOTOROLA", "HUAWEI", "XIAOMI", "OPPO", "REALME", 
              "HONOR", "VIVO", "ALCATEL", "NOKIA", "ZTE", "ONEPLUS", "TCL", "ACER",
              "SENWA", "KODAK", "NUBIA", "CUBOT", "INFINIX", "TECNO", "ZUUM", 
              "POCO", "HISENSE", "LG", "GOOGLE", "SONY", "DOPPIO", "BMOBILE", 
              "EPIK", "WIKO"]
    
    if "IPHONE" in nombre_upper:
        return "APPLE"
    if "GALAXY" in nombre_upper:
        return "SAMSUNG"
    if "MOTO" in nombre_upper:
        return "MOTOROLA"
    if "POCO" in nombre_upper:
        return "XIAOMI"  # POCO es submarca de Xiaomi
    
    for marca in marcas:
        if marca in nombre_upper:
            return marca
    return ""

def extraer_color_capacidad(sku, nombre=""):
    """Extrae color y capacidad usando diccionarios Excel primero, luego del nombre"""
    # Prioridad 1: Buscar en diccionarios por SKU
    sku = normalizar_sku(sku)
    color = sku_color_dict.get(sku, "")
    capacidad = sku_capacidad_dict.get(sku, "")
    
    # Prioridad 2: Si no está en diccionario, extraer del nombre
    if not color and nombre:
        color = extraer_color_desde_nombre(nombre)
    
    if not capacidad and nombre:
        capacidad = extraer_capacidad_desde_nombre(nombre)
    
    return color, capacidad

def extraer_capacidad_desde_nombre(nombre):
    """Extrae la capacidad de almacenamiento del nombre (GB o TB)"""
    if not nombre:
        return ""
    
    capacidades_gb = {16, 32, 64, 128, 256, 512}
    capacidades_tb = {1, 2}
    
    # Buscar TODAS las capacidades (ej. "6gb Ram 128gb", "12 Gb+512 Gb", "1TB", "1 TB")
    # y quedarse con la mayor, para no confundir la RAM con el almacenamiento
    encontradas = []
    for match in re.finditer(r"(?<!\d)(\d{1,4})\s*(GB|TB)(?![A-Z])", nombre, re.IGNORECASE):
        numero = int(match.group(1))
        unidad = match.group(2).upper()
        if unidad == "GB" and numero == 1024:
            numero, unidad = 1, "TB"
        if unidad == "TB" and numero in capacidades_tb:
            encontradas.append((numero * 1024, f"{numero} TB"))
        elif unidad == "GB" and numero in capacidades_gb:
            encontradas.append((numero, f"{numero} GB"))
    
    if encontradas:
        return max(encontradas)[1]
    
    # Buscar números sin unidad (asumir GB)
    match = re.search(r"(?<!\d)(64|128|256|512|1024)(?!\d)", nombre)
    if match:
        return "1 TB" if match.group(1) == "1024" else f"{match.group(1)} GB"
    
    return ""

def extraer_color_desde_nombre(nombre):
    """Extrae color del nombre del producto con formato capitalizado correcto"""
    if not nombre:
        return ""
    
    nombre_clean = nombre.upper().strip()
    
    # Diccionario de patrones de búsqueda con sus respectivos colores formateados
    # Ordenados por especificidad (patrones más largos primero)
    patrones_colores = [
        # Patrones de colores compuestos (buscar exactos primero)
        ("TITANIO DEL DESIERTO", "Titanio del desierto"),
        ("TITANIUM DESERT", "Titanio del desierto"),
        ("ROSA DORADO", "Rosa dorado"),
        ("ROSE GOLD", "Rosa dorado"),
        ("AZUL MARINO", "Azul marino"),
        ("NAVY BLUE", "Azul marino"),
        ("VERDE OLIVA", "Verde oliva"),
        ("OLIVE GREEN", "Verde oliva"),
        ("GRIS ESPACIAL", "Gris espacial"),
        ("SPACE GRAY", "Gris espacial"),
        ("SPACE GREY", "Gris espacial"),
        ("NEGRO MEDIANOCHE", "Negro medianoche"),
        ("MIDNIGHT BLACK", "Negro medianoche"),
        ("AZUL PACIFICO", "Azul pacífico"),
        ("PACIFIC BLUE", "Azul pacífico"),
        ("ROSA ORO", "Rosa oro"),
        ("VERDE ALPINO", "Verde alpino"),
        ("ALPINE GREEN", "Verde alpino"),
        ("PURPURA PROFUNDO", "Púrpura profundo"),
        ("DEEP PURPLE", "Púrpura profundo"),
        ("SIERRA BLUE", "Sierra blue"),
        ("PRODUCT RED", "Product red"),
        
        # Patrones de colores específicos
        ("TITANIUM", "Titanio"),
        ("TITANIO", "Titanio"),
        ("GRAPHITE", "Grafito"),
        ("GRAFITO", "Grafito"),
        ("CHAMPAGNE", "Champagne"),
        ("BRONZE", "Bronce"),
        ("BRONCE", "Bronce"),
        ("CORAL", "Coral"),
        ("LAVENDER", "Lavanda"),
        ("LAVANDA", "Lavanda"),
        ("MINT", "Menta"),
        ("MENTA", "Menta"),
        ("CREAM", "Crema"),
        ("CREMA", "Crema"),
        ("TURQUOISE", "Turquesa"),
        ("TURQUESA", "Turquesa"),
        ("MIDNIGHT", "Midnight"),
        ("STARLIGHT", "Starlight"),
        ("NAVY", "Navy"),
        ("LILA", "Lila"),
        ("LILAC", "Lila"),
        ("CIAN", "Cian"),
        ("CYAN", "Cian"),
        ("CYAN BLUE", "Cian"),
        ("MAGENTA", "Magenta"),
        ("AMBER", "Ámbar"),
        ("AMBAR", "Ámbar"),
        ("PEARL", "Perla"),
        ("PERLA", "Perla"),
        ("IVORY", "Marfil"),
        ("MARFIL", "Marfil"),
        ("SAPPHIRE", "Zafiro"),
        ("ZAFIRO", "Zafiro"),
        ("EMERALD", "Esmeralda"),
        ("ESMERALDA", "Esmeralda"),
        ("RUBY", "Rubí"),
        ("RUBI", "Rubí"),
        ("ONYX", "Ónix"),
        ("ONIX", "Ónix"),
        ("PEACH", "Durazno"),
        ("DURAZNO", "Durazno"),
        ("LIME", "Lima"),
        ("LIMA", "Lima"),
        ("COBALT", "Cobalto"),
        ("COBALTO", "Cobalto"),
        ("TANGERINE", "Mandarina"),
        ("MANDARINA", "Mandarina"),
        ("PLUM", "Ciruela"),
        ("CIRUELA", "Ciruela"),
        ("CHERRY", "Cereza"),
        ("CEREZA", "Cereza"),
        ("PEAR", "Pera"),
        ("PERA", "Pera"),
        
        # Patrones de colores básicos (al final)
        ("BLACK", "Negro"),
        ("NEGRO", "Negro"),
        ("GREEN", "Verde"),
        ("VERDE", "Verde"),
        ("CAFE", "Café"),
        ("Café", "Café"),
        ("BROWN", "Marrón"),
        ("MARRON", "Marrón"),
        ("NEGRO", "Negro"),
        ("BLUE", "Azul"),
        ("AZUL", "Azul"),
        ("WHITE", "Blanco"),
        ("BLANCO", "Blanco"),
        ("GOLD", "Dorado"),
        ("DORADO", "Dorado"),
        ("ORO", "Dorado"),
        ("SILVER", "Plata"),
        ("PLATA", "Plata"),
        ("RED", "Rojo"),
        ("ROJO", "Rojo"),
        ("GRAY", "Gris"),
        ("GREY", "Gris"),
        ("GRIS", "Gris"),
        ("PINK", "Rosa"),
        ("ROSA", "Rosa"),
        ("PURPLE", "Morado"),
        ("MORADO", "Morado"),
        ("VIOLET", "Morado"),
        ("VIOLETA", "Morado"),
        ("YELLOW", "Amarillo"),
        ("AMARILLO", "Amarillo"),
        ("ORANGE", "Naranja"),
        ("NARANJA", "Naranja")
    ]
    
    # Buscar patrones en el texto (orden de especificidad - más largos primero)
    for patron, color_formateado in patrones_colores:
        if patron in nombre_clean:
            return color_formateado
    
    return ""

def get_total_products_categoria(url_base):
    """Obtiene el total de productos de una categoría"""
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url_base)
        time.sleep(8)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # Buscar total de productos
        alert_elements = soup.find_all("p", attrs={"role": "alert"})
        for element in alert_elements:
            text = element.text.strip()
            match = re.search(r'(\d[\d,]*)', text)
            if match:
                return int(match.group(1).replace(',', ''))
        return 0
    except Exception as e:
        print(f"Error obteniendo total: {e}")
        return 0
    finally:
        driver.quit()

def scrape_page_categoria(url_base, begin_index):
    """Hace scraping de una página específica"""
    url = f"{url_base}?beginIndex={begin_index}" if "?" not in url_base else f"{url_base}&beginIndex={begin_index}"
    
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(12)
        
        # Scroll para cargar contenido
        for i in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        products = soup.find_all("div", attrs={"data-testid": lambda v: v and v.startswith("product-")})
        
        page_data = []
        fecha_consulta = datetime.now().strftime("%Y-%m-%d")
        
        for product in products:
            try:
                # Enlace del producto
                link_element = product.find("a", href=True)
                if not link_element:
                    continue
                href = link_element['href'].strip()
                # El sitio a veces entrega la URL ya absoluta y a veces relativa
                if href.startswith("http"):
                    url_producto = href
                else:
                    url_producto = "https://www.coppel.com" + (href if href.startswith("/") else "/" + href)
                
                # Nombre del producto
                nombre_element = product.find("h3", attrs={"data-testid": "product_mosaico_name"})
                nombre_val = nombre_element.text.strip() if nombre_element else ""
                if not nombre_val:
                    continue
                
                # Precios
                precio_descuento = product.find("span", attrs={"data-testid": "product_mosaico_discounted_price"})
                precio_normal = product.find("span", attrs={"data-testid": "product_mosaico_price"})
                
                if precio_descuento and precio_normal:
                    precio_oferta_val = format_price(precio_descuento.text)
                    precio_normal_val = format_price(precio_normal.text)
                    porcentaje_descuento = calcular_porcentaje_descuento(precio_normal_val, precio_oferta_val)
                elif precio_normal:
                    precio_normal_val = format_price(precio_normal.text)
                    precio_oferta_val = ""
                    porcentaje_descuento = "No discount"
                else:
                    continue
                
                # Extraer información
                sku = extraer_sku(url_producto)
                store_val = "Coppel" if "pm-" in url_producto else "Graymarket"
                marca_val = extraer_marca(sku, nombre_val)  # Ahora pasa SKU y nombre
                color_val, capacidad_val = extraer_color_capacidad(sku, nombre_val)
                
                page_data.append({
                    "Store": store_val,
                    "Marca": marca_val,
                    "Nombre": nombre_val,
                    "Color": color_val,
                    "Capacidad": capacidad_val,
                    "SKU": sku,
                    "URL": url_producto,
                    "Fecha_consulta": fecha_consulta,
                    "Precio_normal": precio_normal_val,
                    "Precio_oferta": precio_oferta_val,
                    "Porcentaje_descuento": porcentaje_descuento
                })
                
            except Exception as e:
                continue
        
        return page_data
    except Exception as e:
        print(f"Error en página {begin_index}: {e}")
        return []
    finally:
        driver.quit()

print("✅ Funciones de scraping cargadas")
print("▶️ Continúa con la Celda 3")

🔧 Cargando funciones de scraping...
✅ Funciones de scraping cargadas
▶️ Continúa con la Celda 3


In [3]:
# 🚀 CELDA 3: SCRAPING PARALELO COMPLETO
print("🚀 INICIANDO SCRAPING PARALELO COMPLETO")
print("="*60)

def ejecutar_scraping_completo():
    """Ejecuta scraping paralelo de todas las categorías"""
    
    # Configuración de categorías
    urls_categorias = [
        ("https://www.coppel.com/ct/celulares/celulares-por-compania/telefonia-libre/cat000029", "Graymarket"),
        ("https://www.coppel.com/ct/celulares/celulares-por-compania/celulares-reacondicionados/cat000031", "Graymarket_Reacondicionados"),
        ("https://www.coppel.com/ct/celulares/celulares-por-compania/telcel/cat000027", "Telcel"),
        ("https://www.coppel.com/ct/celulares/celulares-por-compania/movistar/cat000028", "Movistar"),
        ("https://www.coppel.com/ct/celulares/celulares-por-compania/att-unefon/cat000030", "AT&T")
    ]
    
    all_data = []
    data_lock = Lock()
    
    def procesar_pagina_paralelo(args):
        """Procesa una página en paralelo"""
        url_base, begin_index, carrier_val = args
        try:
            page_data = scrape_page_categoria(url_base, begin_index)
            if page_data:
                for prod in page_data:
                    prod["Carrier"] = carrier_val
                with data_lock:
                    all_data.extend(page_data)
                print(f"✅ {carrier_val} p.{begin_index//24+1}: {len(page_data)} productos")
                return len(page_data)
            return 0
        except Exception as e:
            print(f"❌ Error {carrier_val} p.{begin_index//24+1}: {e}")
            return 0
    
    # Preparar tareas
    tareas = []
    for url_base, carrier_val in urls_categorias:
        print(f"\n📍 Analizando {carrier_val}...")
        total_productos = get_total_products_categoria(url_base)
        if total_productos > 0:
            num_paginas = (total_productos // 24) + (1 if total_productos % 24 > 0 else 0)
            print(f"   📊 {total_productos} productos en {num_paginas} páginas")
            
            for i in range(num_paginas):
                tareas.append((url_base, i * 24, carrier_val))
        else:
            print(f"   ⚠️ No se detectaron productos")
    
    print(f"\n🎯 Total tareas: {len(tareas)} páginas")
    print("🔄 Procesando en paralelo...")
    
    # Ejecutar en paralelo
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(procesar_pagina_paralelo, tarea): tarea for tarea in tareas}
        
        productos_total = 0
        for future in concurrent.futures.as_completed(futures):
            try:
                productos_pagina = future.result()
                productos_total += productos_pagina
            except Exception as e:
                print(f"❌ Error en tarea: {e}")
    
    return all_data

# EJECUTAR SCRAPING
resultado_scraping = ejecutar_scraping_completo()

if resultado_scraping:
    df_final = pd.DataFrame(resultado_scraping)
    print(f"\n🎉 ¡SCRAPING EXITOSO!")
    print(f"📊 Total productos: {len(df_final):,}")
    
    # Resumen por carrier
    print(f"\n📊 Por Carrier:")
    for carrier, count in df_final.groupby('Carrier').size().items():
        print(f"  {carrier}: {count:,}")
    
    print(f"\n📊 Top 5 Marcas:")
    for marca, count in df_final.groupby('Marca').size().head().items():
        print(f"  {marca}: {count:,}")
    
    print("✅ Datos listos para exportar")
    print("▶️ Continúa con la Celda 4")
else:
    print("❌ Error en el scraping")

print("="*60)

🚀 INICIANDO SCRAPING PARALELO COMPLETO

📍 Analizando Graymarket...
   📊 1332 productos en 56 páginas

📍 Analizando Graymarket_Reacondicionados...
   📊 559 productos en 24 páginas

📍 Analizando Telcel...
   📊 419 productos en 18 páginas

📍 Analizando Movistar...
   📊 22 productos en 1 páginas

📍 Analizando AT&T...
   📊 58 productos en 3 páginas

🎯 Total tareas: 102 páginas
🔄 Procesando en paralelo...
✅ Graymarket p.3: 24 productos
✅ Graymarket p.1: 24 productos
✅ Graymarket p.4: 24 productos
✅ Graymarket p.2: 24 productos
✅ Graymarket p.7: 24 productos
✅ Graymarket p.5: 24 productos
✅ Graymarket p.6: 24 productos
✅ Graymarket p.8: 24 productos
✅ Graymarket p.10: 24 productos
✅ Graymarket p.9: 24 productos
✅ Graymarket p.11: 24 productos
✅ Graymarket p.12: 24 productos
✅ Graymarket p.13: 24 productos
✅ Graymarket p.14: 24 productos
✅ Graymarket p.15: 24 productos
✅ Graymarket p.16: 24 productos
✅ Graymarket p.17: 24 productos
✅ Graymarket p.18: 24 productos
✅ Graymarket p.20: 24 producto

In [4]:
# 📁 CELDA 4: EXPORTAR CSV FINAL CON FORMATO CORREGIDO
print("📁 PROCESAMIENTO FINAL Y EXPORTACIÓN")
print("="*60)

if 'df_final' in locals() and len(df_final) > 0:
    
    print("🔄 PASO 1: CALCULANDO PORCENTAJES DE DESCUENTO...")
    
    def calcular_porcentaje_descuento_df(precio_normal, precio_oferta):
        """Calcula porcentaje de descuento para DataFrame"""
        try:
            if pd.isna(precio_oferta) or precio_oferta == "" or precio_oferta == "No discount":
                return "No discount"
            
            p_normal = float(str(precio_normal).replace('$','').replace(',','')) if precio_normal else 0
            p_oferta = float(str(precio_oferta).replace('$','').replace(',','')) if precio_oferta else 0
            
            if p_normal > 0 and p_oferta > 0 and p_normal > p_oferta:
                descuento = round(100 * (p_normal - p_oferta) / p_normal, 2)
                return f"{descuento}%"
            elif p_oferta > 0:
                return "0%"
        except Exception:
            pass
        return "No discount"
    
    # Aplicar cálculo de descuentos si no existe la columna
    if 'Porcentaje_descuento' not in df_final.columns:
        df_final['Porcentaje_descuento'] = df_final.apply(
            lambda row: calcular_porcentaje_descuento_df(row['Precio_normal'], row['Precio_oferta']), 
            axis=1
        )
    
    print("🔄 PASO 2: CORRIGIENDO FORMATO DE DATOS...")
    
    # Corregir formato de porcentajes y precios
    def corregir_precio_oferta(row):
        if pd.isna(row['Precio_oferta']) or row['Precio_oferta'] == '' or row['Precio_oferta'] == row['Precio_normal']:
            return 'No discount'
        return row['Precio_oferta']
    
    df_final['Precio_oferta'] = df_final.apply(corregir_precio_oferta, axis=1)
    
    # Asegurar que porcentajes mantengan formato "No discount"
    df_final['Porcentaje_descuento'] = df_final['Porcentaje_descuento'].apply(
        lambda x: 'No discount' if pd.isna(x) or x == '0%' or x == 0 else x
    )
    
    print("🔄 PASO 3: ELIMINANDO DUPLICADOS...")
    
    # Eliminar duplicados
    total_antes = len(df_final)
    duplicados_por_sku = df_final.duplicated(subset=['SKU'], keep='first').sum()
    
    if duplicados_por_sku > 0:
        df_final = df_final.drop_duplicates(subset=['SKU'], keep='first')
    
    # Eliminar duplicados adicionales por Nombre + Marca
    duplicados_restantes = df_final.duplicated(subset=['Nombre', 'Marca'], keep='first').sum()
    if duplicados_restantes > 0:
        df_final = df_final.drop_duplicates(subset=['Nombre', 'Marca'], keep='first')
    
    df_final = df_final.reset_index(drop=True)
    total_despues = len(df_final)
    duplicados_eliminados = total_antes - total_despues
    
    print(f"   • Duplicados eliminados: {duplicados_eliminados:,}")
    print(f"   • Productos únicos: {total_despues:,}")
    
    print("🔄 PASO 4: GENERANDO ARCHIVO CSV FINAL...")
    
    # Crear nombre de archivo con timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archivo_csv_final = f"coppel_celulares_final_{timestamp}.csv"
    
    # Exportar CSV final
    df_final.to_csv(archivo_csv_final, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ ARCHIVO CSV FINAL GENERADO:")
    print(f"📁 Archivo: {archivo_csv_final}")
    print(f"📊 Productos únicos: {len(df_final):,}")
    
    # Estadísticas finales de descuentos
    productos_con_descuento = len(df_final[df_final['Porcentaje_descuento'] != 'No discount'])
    productos_sin_descuento = len(df_final[df_final['Porcentaje_descuento'] == 'No discount'])
    
    print(f"\n📊 ESTADÍSTICAS FINALES:")
    print(f"📱 Total productos únicos: {len(df_final):,}")
    print(f"💰 Productos CON descuento: {productos_con_descuento:,} ({productos_con_descuento/len(df_final)*100:.1f}%)")
    print(f"🏷️ Productos SIN descuento: {productos_sin_descuento:,} ({productos_sin_descuento/len(df_final)*100:.1f}%)")
    
    # Estadísticas por carrier
    carrier_stats = df_final.groupby('Carrier').size().sort_values(ascending=False)
    print(f"\n📊 PRODUCTOS POR CARRIER:")
    for carrier, count in carrier_stats.items():
        porcentaje = (count / len(df_final)) * 100
        print(f"  {carrier}: {count:,} ({porcentaje:.1f}%)")
    
    # Top marcas
    marca_stats = df_final.groupby('Marca').size().sort_values(ascending=False).head(10)
    print(f"\n🏷️ TOP 10 MARCAS:")
    for marca, count in marca_stats.items():
        print(f"  {marca}: {count:,}")
    
    # Análisis de precios
    df_precios = df_final.copy()
    df_precios['Precio_num'] = pd.to_numeric(df_precios['Precio_normal'], errors='coerce')
    precios_validos = df_precios['Precio_num'].dropna()
    
    if len(precios_validos) > 0:
        print(f"\n💰 ANÁLISIS DE PRECIOS:")
        print(f"  Precio promedio: ${precios_validos.mean():,.0f}")
        print(f"  Precio mínimo: ${precios_validos.min():,.0f}")
        print(f"  Precio máximo: ${precios_validos.max():,.0f}")
        print(f"  Precio mediano: ${precios_validos.median():,.0f}")
    
    # Muestra de productos con y sin descuento
    print(f"\n🔍 MUESTRA DE DATOS:")
    if productos_con_descuento > 0:
        print("Productos CON descuento:")
        con_desc = df_final[df_final['Porcentaje_descuento'] != 'No discount'][['Nombre', 'Precio_normal', 'Precio_oferta', 'Porcentaje_descuento']].head(3)
        for _, row in con_desc.iterrows():
            print(f"  • {row['Nombre'][:50]}... - {row['Porcentaje_descuento']} (${row['Precio_normal']} → {row['Precio_oferta']})")
    
    if productos_sin_descuento > 0:
        print("Productos SIN descuento:")
        sin_desc = df_final[df_final['Porcentaje_descuento'] == 'No discount'][['Nombre', 'Precio_normal', 'Precio_oferta', 'Porcentaje_descuento']].head(3)
        for _, row in sin_desc.iterrows():
            print(f"  • {row['Nombre'][:50]}... - {row['Precio_normal']} (Sin descuento)")
    
    print(f"\n📋 COLUMNAS EN EL CSV FINAL:")
    for i, col in enumerate(df_final.columns, 1):
        print(f"  {i:2d}. {col}")
    
    # Verificar archivo generado
    import os
    if os.path.exists(archivo_csv_final):
        tamaño = os.path.getsize(archivo_csv_final) / (1024 * 1024)
        print(f"\n📏 Tamaño del archivo: {tamaño:.2f} MB")
    
    print(f"\n🎉 ¡EXPORTACIÓN FINAL COMPLETADA!")
    print(f"📁 Archivo listo: {archivo_csv_final}")
    print(f"✅ Características del archivo:")
    print(f"   • Sin duplicados")
    print(f"   • Formato 'No discount' para productos sin oferta")
    print(f"   • Porcentajes con símbolo % para productos con descuento")
    print(f"   • Datos completos de marcas, colores y capacidades")
    print(f"   • Listo para análisis en Excel, Power BI, Tableau")
    
else:
    print("❌ ERROR: No hay datos para exportar")
    print("🔄 Ejecuta primero la Celda 3 (Scraping)")

print("="*60)

📁 PROCESAMIENTO FINAL Y EXPORTACIÓN
🔄 PASO 1: CALCULANDO PORCENTAJES DE DESCUENTO...
🔄 PASO 2: CORRIGIENDO FORMATO DE DATOS...
🔄 PASO 3: ELIMINANDO DUPLICADOS...
   • Duplicados eliminados: 230
   • Productos únicos: 2,112
🔄 PASO 4: GENERANDO ARCHIVO CSV FINAL...

✅ ARCHIVO CSV FINAL GENERADO:
📁 Archivo: coppel_celulares_final_20260929_102143.csv
📊 Productos únicos: 2,112

📊 ESTADÍSTICAS FINALES:
📱 Total productos únicos: 2,112
💰 Productos CON descuento: 1,626 (77.0%)
🏷️ Productos SIN descuento: 486 (23.0%)

📊 PRODUCTOS POR CARRIER:
  Graymarket: 1,298 (61.5%)
  Graymarket_Reacondicionados: 528 (25.0%)
  Telcel: 206 (9.8%)
  AT&T: 58 (2.7%)
  Movistar: 22 (1.0%)

🏷️ TOP 10 MARCAS:
  APPLE: 574
  SAMSUNG: 493
  MOTOROLA: 206
  XIAOMI: 179
  HONOR: 141
  : 120
  OPPO: 69
  VIVO: 66
  GOOGLE: 56
  REALME: 50

💰 ANÁLISIS DE PRECIOS:
  Precio promedio: $12,468
  Precio mínimo: $412
  Precio máximo: $86,554
  Precio mediano: $9,799

🔍 MUESTRA DE DATOS:
Productos CON descuento:
  • Celular Xi